# OncoVault — Symptom Leukemia Risk Assessment Model (`leukemia-symptom-v1`)
### Google Colab Training & Academic Validation Notebook

**Dataset Source:** *Symptom Based Explainable Artificial Intelligence Model for Leukemia Detection* (Akter et al., Bangladesh)
- **Training Cohort:** Dhaka Shishu Hospital ($N=709$, 510 leukemia, 199 non-leukemia)
- **Independent Test Cohort:** NICRH ($N=131$, 90 leukemia, 41 non-leukemia)

> **Clinical Note:** This model is designed for Clinical Decision Support and risk screening in pediatric cohorts, not automated definitive diagnosis.

In [ ]:
# 1. Setup Environment
import os, csv, json, joblib
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
print('Environment initialized successfully.')

In [ ]:
# 2. Define 16 Finalized Symptoms
SYMPTOM_COLS = [
    'shortness of breath', 'bone_pain', 'fever', 'family_history',
    'frequent_infections', 'Itchy_skin_or_rash', 'loss_of_appetite_or_nausea',
    'Persistent_weakness _and_fatigue', 'swollen,painless_lymph',
    'significant_bruising,bleeding', 'enlarged_liver', 'oral_cavity',
    'vision_blurring', 'jaundice', 'night_sweats', 'smokes'
]

def load_symptoms(filepath):
    with open(filepath, 'r', encoding='utf-8') as f:
        reader = csv.reader(f)
        header = next(reader)
        col_indices = [header.index(c) for c in SYMPTOM_COLS]
        label_idx = header.index('leukemia')
        X, y = [], []
        for r in reader:
            if not r or not any(cell.strip() for cell in r): continue
            X.append([float(r[i].strip()) for i in col_indices])
            y.append(int(float(r[label_idx].strip())))
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int32)

# Load datasets (adjust paths as needed)
X_train, y_train = load_symptoms('oncovault_ai/data/symptoms/train.csv')
X_test, y_test = load_symptoms('oncovault_ai/data/symptoms/test.csv')
print(f'Training subjects (Dhaka Shishu Hospital): {len(X_train)}')
print(f'Independent test subjects (NICRH):         {len(X_test)}')

In [ ]:
# 3. 5-Fold Stratified Cross Validation on Training Data Only
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
models = {
    'Logistic Regression (Balanced)': LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000),
    'Logistic Regression (Default)':  LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest':                 RandomForestClassifier(n_estimators=100, max_depth=6, class_weight='balanced', random_state=42),
    'Gradient Boosting':             GradientBoostingClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, random_state=42)
}

print(f"{'Model':<30} | {'Accuracy':<10} | {'Sensitivity':<12} | {'Precision':<10} | {'F1':<8} | {'ROC-AUC':<10}")
print('-'*88)
for name, model in models.items():
    accs, recs, precs, f1s, aucs = [], [], [], [], []
    for tr_idx, val_idx in skf.split(X_train, y_train):
        clf = model.__class__(**model.get_params())
        clf.fit(X_train[tr_idx], y_train[tr_idx])
        preds = clf.predict(X_train[val_idx])
        probs = clf.predict_proba(X_train[val_idx])[:, 1]
        accs.append(accuracy_score(y_train[val_idx], preds))
        recs.append(recall_score(y_train[val_idx], preds))
        precs.append(precision_score(y_train[val_idx], preds))
        f1s.append(f1_score(y_train[val_idx], preds))
        aucs.append(roc_auc_score(y_train[val_idx], probs))
    print(f"{name:<30} | {np.mean(accs):.4f}     | {np.mean(recs):.4f}       | {np.mean(precs):.4f}     | {np.mean(f1s):.4f}   | {np.mean(aucs):.4f}")

In [ ]:
# 4. Final Fit and Independent Evaluation on NICRH Test Set
final_symptom_model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, random_state=42)
final_symptom_model.fit(X_train, y_train)

test_preds = final_symptom_model.predict(X_test)
test_probs = final_symptom_model.predict_proba(X_test)[:, 1]
cm = confusion_matrix(y_test, test_preds)
tn, fp, fn, tp = cm.ravel()

print('=== NICRH INDEPENDENT TEST RESULTS ===')
print(f'Accuracy:             {accuracy_score(y_test, test_preds):.4f}')
print(f'Recall / Sensitivity: {recall_score(y_test, test_preds):.4f}')
print(f'Specificity:          {tn/(tn+fp):.4f}')
print(f'Precision:            {precision_score(y_test, test_preds):.4f}')
print(f'F1-Score:             {f1_score(y_test, test_preds):.4f}')
print(f'ROC-AUC:              {roc_auc_score(y_test, test_probs):.4f}')
print(f'Confusion Matrix:     TN={tn}, FP={fp}, FN={fn}, TP={tp}')